# 06. Agents & Tools

## 학습 목표
- LLM Agent의 개념과 구성 요소 이해
- Function Calling으로 도구 사용 구현
- ReAct 패턴 (Thought → Action → Observation) 실습
- 간단한 Agent 직접 구현
- Agent 프레임워크 소개 (LangChain, LlamaIndex)
- Agent의 한계와 주의점

## 핵심 논문
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (Yao et al., 2023)

---

In [ ]:
import json
import re
import math
import random
from datetime import datetime
from typing import Any, Callable, Optional

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np
import pandas as pd

## 1. LLM Agent란: 관찰 → 추론 → 행동 루프

### Agent vs 일반 LLM

| 항목 | 일반 LLM | Agent |
|------|---------|-------|
| **입력** | 텍스트 프롬프트 | 텍스트 + 도구 + 환경 |
| **출력** | 텍스트 응답 | 행동 (API 호출, 파일 작성 등) |
| **상호작용** | 단발성 | 다단계 루프 |
| **외부 정보** | 사전 학습된 지식만 | 도구로 실시간 정보 획득 |

### Agent 구성 요소

```
+---------------------+
|       Agent         |
|  +---------------+  |
|  |     LLM       |  |  \u2190 \ub450\ub1cc (\ucd94\ub860/\uacc4\ud68d)
|  +---------------+  |
|  | Tools/Actions |  |  \u2190 \ub3c4\uad6c (\uacc4\uc0b0\uae30, \uac80\uc0c9, API)
|  +---------------+  |
|  |    Memory      |  |  \u2190 \uae30\uc5b5 (\ub300\ud654 \uc774\ub825, \uc911\uac04 \uacb0\uacfc)
|  +---------------+  |
+---------------------+
```

---
## 2. Function Calling: 함수 정의, LLM이 호출, 결과 반환

Agent의 가장 기본적인 기능: LLM이 **어떤 함수를 어떤 인자로 호출할지** 결정.

### 흐름

```
\uc0ac\uc6a9\uc790: "\uc11c\uc6b8 \ub0a0\uc528 \uc54c\ub824\uc918"
    \u2193
LLM: get_weather(city="\uc11c\uc6b8") \ud638\ucd9c \uacb0\uc815
    \u2193
\uc2dc\uc2a4\ud15c: get_weather \uc2e4\ud589 \u2192 {"temp": 22, "condition": "\ub9d1\uc74c"}
    \u2193
LLM: "\uc11c\uc6b8\uc758 \ud604\uc7ac \uae30\uc628\uc740 22\ub3c4\uc774\uace0 \ub0a0\uc528\ub294 \ub9d1\uc2b5\ub2c8\ub2e4."
```

In [ ]:
# --- 도구(Tool) 정의 ---

def calculator(expression: str) -> str:
    """수학 계산을 수행합니다."""
    try:
        # 안전한 수식 평가 (기본 연산만)
        allowed = set('0123456789+-*/().% ')
        if not all(c in allowed for c in expression):
            return f"Error: 허용되지 않는 문자가 포함되어 있습니다."
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"


def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다. (Mock)"""
    weather_data = {
        "서울": {"temp": 22, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 25, "condition": "구름 조금", "humidity": 60},
        "제주": {"temp": 20, "condition": "비", "humidity": 80},
    }
    data = weather_data.get(city)
    if data:
        return json.dumps(data, ensure_ascii=False)
    return f"\"{city}\" 날씨 정보를 찾을 수 없습니다."


def search_web(query: str) -> str:
    """웹 검색을 수행합니다. (Mock)"""
    search_results = {
        "성수동 카페": [
            {"title": "성수동 카페 BEST 10", "snippet": "블루보틀, 컨테이너, 솔길체 등"},
            {"title": "성수동 맛집 추천", "snippet": "성수동 맛집 리스트"}
        ],
        "한국 인구": [
            {"title": "한국 인구 통계", "snippet": "2024년 한국 인구는 약 5,180만 명"}
        ]
    }
    # 쿼리 키워드 매칭
    for key, results in search_results.items():
        if key in query:
            return json.dumps(results, ensure_ascii=False)
    return json.dumps([{"title": "검색 결과 없음", "snippet": f"\"{query}\"\uc5d0 \ub300\ud55c \uacb0\uacfc\ub97c \ucc3e\uc744 \uc218 \uc5c6\uc2b5\ub2c8\ub2e4."}], ensure_ascii=False)


def get_current_time() -> str:
    """현재 시간을 반환합니다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# 도구 레지스트리
TOOLS = {
    "calculator": {
        "function": calculator,
        "description": "수학 계산을 수행합니다. 입력: 수식 문자열",
        "parameters": {"expression": "string"}
    },
    "get_weather": {
        "function": get_weather,
        "description": "도시의 현재 날씨를 조회합니다. 입력: 도시명",
        "parameters": {"city": "string"}
    },
    "search_web": {
        "function": search_web,
        "description": "웹 검색을 수행합니다. 입력: 검색 쿼리",
        "parameters": {"query": "string"}
    },
    "get_current_time": {
        "function": get_current_time,
        "description": "현재 시간을 반환합니다. 입력: 없음",
        "parameters": {}
    }
}

print("=== 사용 가능한 도구 ===")
for name, tool in TOOLS.items():
    print(f"  {name}: {tool['description']}")

# 도구 테스트
print("\n=== 도구 테스트 ===")
print(f"calculator('15000 * 3 * 0.85'): {calculator('15000 * 3 * 0.85')}")
print(f"get_weather('\uc11c\uc6b8'): {get_weather('\uc11c\uc6b8')}")
print(f"get_current_time(): {get_current_time()}")

---
## 3. ReAct 패턴: Thought → Action → Observation

ReAct (Reasoning + Acting)는 LLM이 **생각하고(Thought)**, **행동하고(Action)**, **관찰하는(Observation)** 루프를 반복하는 패턴.

### ReAct 프레임워크

```
Question: \uc0ac\uc6a9\uc790 \uc9c8\ubb38

Thought 1: \uc774 \uc9c8\ubb38\uc5d0 \ub2f5\ud558\ub824\uba74 \ub0a0\uc528 \uc815\ubcf4\uac00 \ud544\uc694\ud558\ub2e4.
Action 1: get_weather(city="\uc11c\uc6b8")
Observation 1: {"temp": 22, "condition": "\ub9d1\uc74c"}

Thought 2: \ub0a0\uc528 \uc815\ubcf4\ub97c \uc5bb\uc5c8\ub2e4. \uc774\uc81c \ub2f5\ubcc0\ud560 \uc218 \uc788\ub2e4.
Action 2: finish(answer="\uc11c\uc6b8\uc740 22\ub3c4, \ub9d1\uc2b5\ub2c8\ub2e4.")
```

### 핵심 논문: "ReAct" (Yao et al., 2023)

- **Reasoning only** (CoT): 생각만 하고 행동하지 않음 → 외부 정보 활용 불가
- **Acting only**: 맹목적으로 행동 → 비효율적
- **ReAct**: 생각 + 행동 조합 → 최적

In [ ]:
# --- ReAct Agent 구현 ---

class ReActAgent:
    """
    ReAct 패턴 Agent.
    Thought → Action → Observation 루프를 반복.
    """
    
    def __init__(self, tools: dict, max_steps: int = 5):
        self.tools = tools
        self.max_steps = max_steps
        self.history = []  # 실행 기록
    
    def _mock_llm_decide(self, question: str, observations: list) -> dict:
        """
        Mock LLM: 질문과 이전 관찰을 바탕으로 다음 행동 결정.
        실제로는 LLM이 이 결정을 수행.
        """
        q = question.lower()
        step = len(observations)
        
        # 날씨 관련 질문
        if "날씨" in q or "기온" in q:
            city = "서울"  # 기본값
            for c in ["서울", "부산", "제주"]:
                if c in q:
                    city = c
                    break
            
            if step == 0:
                return {
                    "thought": f"{city}의 날씨 정보가 필요하다. get_weather 도구를 사용하자.",
                    "action": "get_weather",
                    "action_input": {"city": city}
                }
            else:
                weather = observations[-1]
                return {
                    "thought": f"날씨 정보를 얻었다. 이제 답변할 수 있다.",
                    "action": "finish",
                    "action_input": {"answer": f"{city}의 날씨: {weather}"}
                }
        
        # 계산 관련 질문
        if "계산" in q or "가격" in q or "달러" in q:
            if step == 0:
                # 수식 추출 시도
                expr = "1200 * 5"  # 기본값
                if "커피" in q and "3잔" in q:
                    expr = "5000 * 3"
                elif "환율" in q or "달러" in q:
                    expr = "100 * 1350"  # $100 → KRW
                return {
                    "thought": "계산이 필요하다. calculator 도구를 사용하자.",
                    "action": "calculator",
                    "action_input": {"expression": expr}
                }
            else:
                return {
                    "thought": "계산 결과를 얻었다.",
                    "action": "finish",
                    "action_input": {"answer": f"계산 결과: {observations[-1]}"}
                }
        
        # 검색 관련 질문
        if "카페" in q or "맛집" in q or "검색" in q or "인구" in q:
            if step == 0:
                search_query = question
                return {
                    "thought": "최신 정보가 필요하다. 웹 검색을 수행하자.",
                    "action": "search_web",
                    "action_input": {"query": search_query}
                }
            else:
                return {
                    "thought": "검색 결과를 얻었다.",
                    "action": "finish",
                    "action_input": {"answer": f"검색 결과: {observations[-1]}"}
                }
        
        # 기본: 직접 답변
        return {
            "thought": "도구 없이 답변할 수 있다.",
            "action": "finish",
            "action_input": {"answer": f"\"{question}\"\uc5d0 \ub300\ud55c \ub2f5\ubcc0\uc785\ub2c8\ub2e4."}
        }
    
    def run(self, question: str) -> dict:
        """
        ReAct 루프 실행.
        """
        self.history = []
        observations = []
        
        print(f"Question: {question}")
        print("-" * 50)
        
        for step in range(self.max_steps):
            # LLM 결정
            decision = self._mock_llm_decide(question, observations)
            
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {decision['thought']}")
            print(f"  Action: {decision['action']}({json.dumps(decision['action_input'], ensure_ascii=False)})")
            
            # 종료 조건
            if decision["action"] == "finish":
                answer = decision["action_input"]["answer"]
                print(f"  \u2192 Final Answer: {answer}")
                self.history.append({"step": step + 1, **decision, "observation": "DONE"})
                return {"answer": answer, "steps": step + 1, "history": self.history}
            
            # 도구 실행
            tool_name = decision["action"]
            if tool_name in self.tools:
                tool_fn = self.tools[tool_name]["function"]
                tool_input = decision["action_input"]
                observation = tool_fn(**tool_input)
            else:
                observation = f"Error: Tool '{tool_name}' not found"
            
            print(f"  Observation: {observation}")
            observations.append(observation)
            self.history.append({"step": step + 1, **decision, "observation": observation})
        
        return {"answer": "Max steps reached", "steps": self.max_steps, "history": self.history}

In [ ]:
# ReAct Agent 실행

agent = ReActAgent(TOOLS, max_steps=5)

# 테스트 1: 날씨 질문
print("=" * 60)
result1 = agent.run("서울 날씨 알려줘")

print("\n" + "=" * 60)
# 테스트 2: 계산 질문
result2 = agent.run("커피 3잔 가격 계산해줘")

print("\n" + "=" * 60)
# 테스트 3: 검색 질문
result3 = agent.run("성수동 카페 추천해줘")

---
## 4. 도구 정의: 스키마와 함수 등록

Agent에게 제공할 도구를 체계적으로 정의하는 방법.

In [ ]:
# --- Tool Schema 정의 (OpenAI Function Calling 형식) ---

tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "수학 계산을 수행합니다. 사칙연산, 백분율 계산 등을 지원합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수식. 예: '15000 * 3 * 0.85'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "도시의 현재 날씨를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "도시명. 예: '서울', '부산'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "웹 검색을 수행하여 최신 정보를 찾습니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "검색할 쿼리 문자열"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("=== Tool Schemas (OpenAI \ud615\uc2dd) ===")
for schema in tool_schemas:
    fn = schema["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)}): {fn['description'][:50]}")

---
## 5. 간단한 Agent 구현: 멀티스텝 문제 해결

여러 도구를 조합해야 하는 복잡한 질문에 대한 Agent 구현.

In [ ]:
# --- 멀티스텝 Agent ---

class MultiStepAgent:
    """여러 도구를 조합하는 멀티스텝 Agent"""
    
    def __init__(self, tools: dict):
        self.tools = tools
        self.memory = []  # 실행 기록
    
    def plan(self, question: str) -> list[dict]:
        """
        질문을 분석하여 실행 계획 수립.
        실제로는 LLM이 계획을 생성.
        """
        q = question.lower()
        steps = []
        
        if "날씨" in q and "가격" in q:
            # 날씨 + 계산 복합 질문
            city = "서울"
            for c in ["서울", "부산", "제주"]:
                if c in q:
                    city = c
            steps = [
                {"thought": f"먼저 {city} 날씨를 확인하자.",
                 "action": "get_weather", "input": {"city": city}},
                {"thought": "그리고 가격을 계산하자.",
                 "action": "calculator", "input": {"expression": "5000 * 3"}},
                {"thought": "모든 정보를 종합하여 답변하자.",
                 "action": "finish", "input": {}}
            ]
        elif "검색" in q and "계산" in q:
            steps = [
                {"thought": "먼저 검색으로 정보를 얻자.",
                 "action": "search_web", "input": {"query": question}},
                {"thought": "검색 결과를 바탕으로 계산하자.",
                 "action": "calculator", "input": {"expression": "5180 * 10000"}},
                {"thought": "결과를 종합하자.",
                 "action": "finish", "input": {}}
            ]
        else:
            # 단순 질문
            steps = [
                {"thought": "직접 답변할 수 있다.",
                 "action": "finish", "input": {}}
            ]
        
        return steps
    
    def execute(self, question: str) -> dict:
        """계획 수립 후 실행"""
        self.memory = []
        plan = self.plan(question)
        results = []
        
        print(f"Question: {question}")
        print(f"Plan: {len(plan)} steps")
        print("-" * 50)
        
        for i, step in enumerate(plan):
            print(f"\nStep {i+1}:")
            print(f"  Thought: {step['thought']}")
            
            if step["action"] == "finish":
                # 모든 결과 종합
                summary = " | ".join(results)
                answer = f"종합 결과: {summary}"
                print(f"  \u2192 Final Answer: {answer}")
                return {"answer": answer, "steps": len(plan), "details": results}
            
            if step["action"] in self.tools:
                tool_fn = self.tools[step["action"]]["function"]
                result = tool_fn(**step["input"])
                print(f"  Action: {step['action']}({step['input']})")
                print(f"  Result: {result}")
                results.append(f"{step['action']}: {result}")
                self.memory.append({"step": i+1, "action": step["action"], "result": result})
        
        return {"answer": "Plan executed", "details": results}


# 멀티스텝 테스트
multi_agent = MultiStepAgent(TOOLS)

print("=" * 60)
result = multi_agent.execute("서울 날씨 확인하고 커피 3잔 가격 계산해줘")

In [ ]:
# Agent 실행 흐름 시각화

def visualize_agent_flow(history: list):
    """에이전트 실행 흐름 시각화"""
    fig, ax = plt.subplots(figsize=(12, 4))
    
    step_names = []
    step_types = []
    
    for i, entry in enumerate(history):
        step_names.append(f"Step {i+1}\n{entry.get('action', 'think')}")
        if entry.get('action') == 'finish':
            step_types.append('finish')
        elif 'observation' in entry:
            step_types.append('action')
        else:
            step_types.append('thought')
    
    colors = {'thought': '#45b7d1', 'action': '#4ecdc4', 'finish': '#f7dc6f'}
    bar_colors = [colors.get(t, 'gray') for t in step_types]
    
    x = np.arange(len(step_names))
    bars = ax.bar(x, [1] * len(step_names), color=bar_colors, edgecolor='black', linewidth=1)
    
    # 화살표 추가
    for i in range(len(step_names) - 1):
        ax.annotate('', xy=(i + 1, 0.5), xytext=(i, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    ax.set_xticks(x)
    ax.set_xticklabels(step_names, fontsize=10)
    ax.set_ylim(0, 1.5)
    ax.set_yticks([])
    ax.set_title("Agent Execution Flow", fontsize=14, fontweight='bold')
    
    # 범례
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#45b7d1', label='Thought'),
        Patch(facecolor='#4ecdc4', label='Action'),
        Patch(facecolor='#f7dc6f', label='Finish'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()


# 시각화
sample_history = [
    {"step": 1, "action": "get_weather", "observation": '{"temp": 22}'},
    {"step": 2, "action": "calculator", "observation": '15000'},
    {"step": 3, "action": "finish", "observation": "DONE"},
]

visualize_agent_flow(sample_history)

---
## 6. Agent 프레임워크: LangChain / LlamaIndex 소개

### LangChain

```python
# LangChain Agent \uc608\uc2dc (\uac1c\ub150 \ucf54\ub4dc)
from langchain.agents import initialize_agent, Tool
from langchain.llms import OpenAI

tools = [
    Tool(name="Calculator", func=calculator, description="\uc218\ud559 \uacc4\uc0b0"),
    Tool(name="Weather", func=get_weather, description="\ub0a0\uc528 \uc870\ud68c"),
]

agent = initialize_agent(
    tools, 
    OpenAI(temperature=0),
    agent="zero-shot-react-description"
)

result = agent.run("\uc11c\uc6b8 \ub0a0\uc528 \uc54c\ub824\uc918")
```

### LlamaIndex

```python
# LlamaIndex Agent \uc608\uc2dc (\uac1c\ub150 \ucf54\ub4dc)
from llama_index.agent import ReActAgent
from llama_index.tools import FunctionTool

tools = [
    FunctionTool.from_defaults(fn=calculator, name="calculator"),
    FunctionTool.from_defaults(fn=get_weather, name="weather"),
]

agent = ReActAgent.from_tools(tools, llm=llm)
response = agent.chat("\uc11c\uc6b8 \ub0a0\uc528 \uc54c\ub824\uc918")
```

### 프레임워크 비교

| 항목 | LangChain | LlamaIndex |
|------|-----------|------------|
| **강점** | 다양한 Agent 패턴 | RAG 특화 |
| **생태계** | 기대 | 성장 중 |
| **학습 곡선** | 중간 | 쉽움 |
| **적합 용도** | 범용 Agent | 데이터 Q&A |

In [ ]:
# 프레임워크 없이 구현한 Agent vs 프레임워크 비교

comparison_data = {
    "Feature": [
        "Setup Complexity",
        "Flexibility",
        "Dependencies",
        "Learning Curve",
        "Production Ready",
        "Debugging",
    ],
    "Custom (this notebook)": [1, 5, 1, 2, 2, 5],
    "LangChain": [3, 4, 4, 3, 4, 3],
    "LlamaIndex": [2, 3, 3, 2, 4, 3],
}

df = pd.DataFrame(comparison_data)
print("=== Agent \uad6c\ud604 \ubc29\uc2dd \ube44\uad50 (1=\ub0ae\uc74c, 5=\ub192\uc74c) ===")
print(df.to_string(index=False))

# 레\uc774\ub354 \ucc28\ud2b8
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = df["Feature"].tolist()
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for col, color in [("Custom (this notebook)", "#ff6b6b"), 
                    ("LangChain", "#4ecdc4"),
                    ("LlamaIndex", "#45b7d1")]:
    values = df[col].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=col, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=9)
ax.set_ylim(0, 6)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title("Agent Implementation Comparison", size=14, fontweight='bold', y=1.08)

plt.tight_layout()
plt.show()

---
## 7. Agent의 한계와 주의점

### 주요 한계

| 한계 | 설명 | 대응 |
|------|------|------|
| **무한 루프** | Agent가 같은 행동을 반복 | max_steps 제한 |
| **잘못된 도구 선택** | 엉뚝한 도구 호출 | 도구 설명 개선, Few-shot |
| **비용 폭발** | 매 단계마다 LLM 호출 | 예산 제한, 캠싱 |
| **안전성** | 위험한 도구 실행 가능 | 도구 권한 제한, 검증 |
| **Hallucination** | 잘못된 파라미터로 도구 호출 | 입력 검증, 샌드박싱 |
| **재현성** | 동일 입력에 다른 결과 | temperature=0, 로그 기록 |

In [ ]:
# --- Agent 안전성 가이드라인 ---

class SafeAgent(ReActAgent):
    """안전 가드레일이 추가된 Agent"""
    
    def __init__(self, tools: dict, max_steps: int = 5, 
                 max_cost: float = 1.0, allowed_tools: list = None):
        super().__init__(tools, max_steps)
        self.max_cost = max_cost
        self.current_cost = 0.0
        self.allowed_tools = allowed_tools or list(tools.keys())
        self.cost_per_call = 0.01  # 호출당 비용
    
    def _validate_action(self, action: str, action_input: dict) -> tuple[bool, str]:
        """도구 호출 전 검증"""
        # 1. 허용된 도구인지 확인
        if action not in self.allowed_tools and action != "finish":
            return False, f"Tool '{action}' is not allowed"
        
        # 2. 비용 한도 확인
        if self.current_cost + self.cost_per_call > self.max_cost:
            return False, f"Cost limit exceeded (${self.current_cost:.2f} / ${self.max_cost:.2f})"
        
        # 3. 입력 검증 (calculator의 경우 위험한 코드 차단)
        if action == "calculator" and "expression" in action_input:
            expr = action_input["expression"]
            if any(keyword in expr for keyword in ["import", "exec", "eval", "os.", "sys."]):
                return False, f"Dangerous expression blocked: {expr}"
        
        return True, "OK"
    
    def run(self, question: str) -> dict:
        """\uc548\uc804 \uac00\ub4dc\ub808\uc77c \uc801\uc6a9\ub41c \uc2e4\ud589"""
        self.current_cost = 0.0
        self.history = []
        observations = []
        
        print(f"Question: {question}")
        print(f"Safety: max_cost=${self.max_cost}, allowed_tools={self.allowed_tools}")
        print("-" * 50)
        
        for step in range(self.max_steps):
            decision = self._mock_llm_decide(question, observations)
            self.current_cost += self.cost_per_call
            
            # 검증
            valid, msg = self._validate_action(decision["action"], decision["action_input"])
            
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {decision['thought']}")
            
            if not valid:
                print(f"  BLOCKED: {msg}")
                return {"answer": f"Action blocked: {msg}", "steps": step + 1}
            
            print(f"  Action: {decision['action']}")
            
            if decision["action"] == "finish":
                answer = decision["action_input"]["answer"]
                print(f"  \u2192 Final: {answer}")
                print(f"  Cost: ${self.current_cost:.2f}")
                return {"answer": answer, "steps": step + 1, "cost": self.current_cost}
            
            tool_fn = self.tools[decision["action"]]["function"]
            observation = tool_fn(**decision["action_input"])
            print(f"  Observation: {observation}")
            observations.append(observation)
        
        return {"answer": "Max steps reached", "cost": self.current_cost}


# 안전 Agent 테스트
safe_agent = SafeAgent(
    TOOLS, 
    max_steps=3, 
    max_cost=0.05,
    allowed_tools=["calculator", "get_weather"]  # search_web 차단
)

print("=== 안전 Agent: 날씨 질문 (\ud5c8\uc6a9) ===")
result1 = safe_agent.run("서울 날씨 알려줘")

print("\n" + "=" * 60)
print("\n=== 안전 Agent: 검색 질문 (\ucc28\ub2e8) ===")
result2 = safe_agent.run("성수동 카페 검색해줘")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 새 도구 추가 및 Agent 확장

아래 요구사항에 맞는 도구를 만들고 Agent에 등록하세요.

1. **unit_converter**: 단위 변환 도구
   - 입력: `{"value": 100, "from": "USD", "to": "KRW"}`
   - 지원 단위: USD/KRW, km/miles, kg/lbs, celsius/fahrenheit

2. Agent에 등록하고 "차 100달러는 원화로 얼마야?" 질문 테스트

In [ ]:
# TODO: unit_converter 도구를 구현하고 Agent에 등록하세요
# 요구사항:
# 1. unit_converter 함수 구현 (\ud658\uc728 \ub370\uc774\ud130 \ud558\ub4dc\ucf54\ub529)
# 2. TOOLS 딕\uc154\ub108\ub9ac\uc5d0 \ucd94\uac00
# 3. ReActAgent\uc758 _mock_llm_decide\uc5d0 \ub2e8\uc704 \ubcc0\ud658 \ucffc\ub9ac \ucc98\ub9ac \ucd94\uac00
# 4. \ud14c\uc2a4\ud2b8

# def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
#     TODO
#     return ""


---
## 핵심 정리

| 개념 | 설명 | 활용 |
|------|------|------|
| Agent | 관찰-추론-행동 루프 | 자동화된 작업 수행 |
| Function Calling | LLM이 함수 호출 결정 | 구조화된 외부 상호작용 |
| ReAct | Thought-Action-Observation | 최적의 Agent 패턴 |
| Tools | Agent가 사용하는 외부 기능 | 계산, 검색, API 등 |
| Safety | 도구 권한, 비용 제한 | 프로덕션 필수 |
| LangChain/LlamaIndex | Agent 프레임워크 | 빠른 프로토타이핑 |

---

### 시리즈 완료

**07-applications** 시리즈의 모든 노트북을 완료했습니다.

1. [01-prompt-engineering.ipynb](01-prompt-engineering.ipynb) - Prompt Engineering
2. [02-structured-extraction.ipynb](02-structured-extraction.ipynb) - 구조화된 추출
3. [03-multi-llm-orchestration.ipynb](03-multi-llm-orchestration.ipynb) - 멀티 LLM
4. [04-rag-basics.ipynb](04-rag-basics.ipynb) - RAG
5. [05-evaluation-pipeline.ipynb](05-evaluation-pipeline.ipynb) - 평가 파이프라인
6. [06-agents-and-tools.ipynb](06-agents-and-tools.ipynb) - Agents & Tools